# FJSP diffusion: machine assignment and priority

The model takes an FJSP instance and generates a schedule. For every operation it generates two
discrete choices:

- **machine**: one of the operation's eligible machines
- **priority**: the operation's start time divided by the makespan, cut into `PRIORITY_BINS` bins

Both are trained with discrete (categorical) diffusion against CP-SAT schedules. To turn a sample
into a schedule, we fix the machines and run list scheduling: repeatedly take the job whose next
operation has the lowest priority and start it as soon as its job and machine are free. This
always gives a feasible schedule, and with CP-SAT's machines and start times it rebuilds the CP-SAT
schedule.

The network works on the operation-machine graph used by Song et al. and DANIEL: operation nodes,
machine nodes, job-order edges, and one edge per eligible (operation, machine) pair.

**Before running:** Runtime > Change runtime type > GPU.

## Config

Every setting of a run is in `CONFIG`. It is saved as `runs/<RUN_NAME>/config.json` and inside
`best.pt`, so each result folder records how it was made. Change a value here, not further down.

In [ ]:
CONFIG = {
    "RUN_NAME": "machine_priority_10x5",
    "TRAIN_SETS": ["SD_original/10x5"],
    "EVAL_SETS": [
        "SD_original/10x5/test",
        "SD_original/8x4/test",
        "SD1_lit/10x5",
        "SD2_lit/10x5+mix",
        "BenchData/Brandimarte",
    ],

    # model
    "HIDDEN": 128,
    "LAYERS": 8,
    "PRIORITY_BINS": 64,          # start time / makespan is cut into this many bins

    # diffusion
    "T": 1000,                    # noise levels in training
    "SAMPLE_STEPS": 50,           # denoising steps when generating

    # training
    "EPOCHS": 100,
    "BATCH_SIZE": 32,             # instances per step
    "LR": 5e-4,
    "WEIGHT_DECAY": 1e-4,
    "PRIORITY_WEIGHT": 1.0,       # loss = machine loss + PRIORITY_WEIGHT * priority loss
    "EVAL_EVERY": 10,             # epochs between validation checks (always with the "current" decoder)
    "VAL_SAMPLES": 4,
    "VAL_LIMIT": 50,              # validation instances used in each check

    # evaluation
    "SAMPLES": 16,                # schedules generated per test instance; we report the first and the best
    "DECODERS": ["current", "nondelay"],   # "current": list scheduling, "nondelay": no idle machine

    "SEED": 0,
}

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install ortools

## Settings

`TRAIN_SETS` are folders under `data/` that contain `train/` and `val/`. Training uses only
instances that have a CP-SAT result under `CP-SAT/` with the same path.

`EVAL_SETS` are folders evaluated at the end. The gap is measured against the CP-SAT result
when one exists.

In [ ]:
import csv
import glob
import importlib.util
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT = "/content/drive/MyDrive/IDL/Project/IDL_Project"
DATA_ROOT = os.path.join(PROJECT, "data")
LABEL_ROOT = os.path.join(PROJECT, "CP-SAT")

globals().update(CONFIG)    # every CONFIG entry becomes a variable: RUN_NAME, EPOCHS, ...
OUT_DIR = os.path.join(PROJECT, "runs", RUN_NAME)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump({**CONFIG, "DEVICE": DEVICE}, f, indent=1)
print("device:", DEVICE)
print("project folder exists:", os.path.isdir(PROJECT))

## Reading instances and CP-SAT results

`.fjs` machine numbers start at 1. Inside this notebook machines start at 0.

In [ ]:
def fjs_files(folder):
    return sorted(glob.glob(os.path.join(folder, "**", "*.fjs"), recursive=True))


def label_path(fjs_path):
    rel = os.path.relpath(fjs_path, DATA_ROOT)
    return os.path.join(LABEL_ROOT, os.path.splitext(rel)[0] + ".json")


def read_fjs(path):
    with open(path) as f:
        lines = [line for line in f.read().strip().splitlines() if line.strip()]
    head = lines[0].split()
    n_jobs, n_machines = int(float(head[0])), int(float(head[1]))
    tokens = iter(" ".join(lines[1:]).split())
    jobs = []
    for _ in range(n_jobs):
        job = []
        for _ in range(int(next(tokens))):
            options = []
            for _ in range(int(next(tokens))):
                machine = int(next(tokens)) - 1
                options.append((machine, int(next(tokens))))
            job.append(options)
        jobs.append(job)
    return {"name": os.path.splitext(os.path.basename(path))[0], "n_machines": n_machines, "jobs": jobs}


def read_label(fjs_path):
    path = label_path(fjs_path)
    if not os.path.exists(path):
        return None
    with open(path) as f:
        label = json.load(f)
    if label.get("schedule") is None:
        return None
    return label

## Optional: solve missing train and val labels with CP-SAT

Training needs one CP-SAT schedule per training instance. This cell reuses
`scripts/solve_instances.py` and writes results to the same `CP-SAT/` folder, so each instance is
solved only once. You can also run that script locally with
`DATA_FOLDER = "SD_original/10x5/train"`, which is usually faster than Colab's CPUs.

A `FEASIBLE` result (time limit reached) is still a good training target.

In [ ]:
SOLVE_MISSING = False
SOLVE_TIME_LIMIT = 30   # seconds per instance

if SOLVE_MISSING:
    spec = importlib.util.spec_from_file_location(
        "solve_instances", os.path.join(PROJECT, "scripts", "solve_instances.py"))
    solver = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(solver)
    solver.TIME_LIMIT = SOLVE_TIME_LIMIT
    for train_set in TRAIN_SETS:
        for split in ("train", "val"):
            paths = [p for p in fjs_files(os.path.join(DATA_ROOT, train_set, split))
                     if not os.path.exists(label_path(p))]
            print(f"{train_set}/{split}: {len(paths)} instances to solve")
            for i, path in enumerate(paths):
                solution = solver.solve(solver.read_fjs(path))
                out = label_path(path)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with open(out, "w") as f:
                    json.dump(solution, f)
                if (i + 1) % 50 == 0:
                    print(f"  {i + 1}/{len(paths)}")

## Instance graph

Operations are numbered job by job. Features are scaled by the instance's mean processing time,
so instances with times 1-20 and 1-99 look alike to the network.

| part | features |
|---|---|
| operation node | position in job, operations left in job, mean and min processing time, share of machines eligible, work left in job |
| machine node | eligible operations per machine (relative), eligible processing time (relative) |
| operation-machine edge | processing time, processing time relative to the operation's fastest machine |
| job edge | operation -> next operation of the same job |

In [ ]:
def build_graph(instance, label=None):
    jobs, n_machines = instance["jobs"], instance["n_machines"]
    op_job, op_pos, options, mean_p, min_p = [], [], [], [], []
    edge_op, edge_m, edge_p = [], [], []
    job_first, job_len = [], []
    for j, job in enumerate(jobs):
        job_first.append(len(op_job))
        job_len.append(len(job))
        for k, opts in enumerate(job):
            o = len(op_job)
            op_job.append(j)
            op_pos.append(k)
            options.append(dict(opts))
            times = [p for _, p in opts]
            mean_p.append(sum(times) / len(times))
            min_p.append(min(times))
            for m, p in opts:
                edge_op.append(o)
                edge_m.append(m)
                edge_p.append(p)
    n = len(op_job)
    scale = sum(mean_p) / n
    max_len = max(job_len)

    work_after = [0.0] * n
    for o in range(n - 2, -1, -1):
        if op_job[o + 1] == op_job[o]:
            work_after[o] = work_after[o + 1] + mean_p[o + 1]
    n_elig = [len(options[o]) for o in range(n)]

    op_x = [[op_pos[o] / max_len,
             (job_len[op_job[o]] - 1 - op_pos[o]) / max_len,
             mean_p[o] / scale,
             min_p[o] / scale,
             n_elig[o] / n_machines,
             work_after[o] / (scale * max_len)] for o in range(n)]

    m_count = [0] * n_machines
    m_load = [0.0] * n_machines
    for o, m, p in zip(edge_op, edge_m, edge_p):
        m_count[m] += 1
        m_load[m] += p
    total_load = sum(m_load)
    # 1.0 means the machine has an average share of the eligible operations / processing time
    m_x = [[m_count[m] / len(edge_op) * n_machines,
            m_load[m] / total_load * n_machines] for m in range(n_machines)]
    edge_x = [[p / scale, p / min_p[o]] for o, p in zip(edge_op, edge_p)]
    job_edges = [(o, o + 1) for o in range(n - 1) if op_job[o] == op_job[o + 1]]

    g = {
        "name": instance["name"], "n": n, "M": n_machines, "E": len(edge_op),
        "op_job": op_job, "options": options, "job_first": job_first, "job_len": job_len,
        "op_x": torch.tensor(op_x, dtype=torch.float),
        "m_x": torch.tensor(m_x, dtype=torch.float),
        "edge_x": torch.tensor(edge_x, dtype=torch.float),
        "edge_op": torch.tensor(edge_op, dtype=torch.long),
        "edge_m": torch.tensor(edge_m, dtype=torch.long),
        "n_elig": torch.tensor(n_elig, dtype=torch.float),
        "job_src": torch.tensor([a for a, _ in job_edges], dtype=torch.long),
        "job_dst": torch.tensor([b for _, b in job_edges], dtype=torch.long),
        "ref_makespan": None, "ref_status": None,
        "target_edge": None, "target_bin": None, "ref_start": None,
    }

    if label is not None:
        machines = [m - 1 for job in label["schedule"] for m, _ in job]
        starts = [s for job in label["schedule"] for _, s in job]
        edge_index = {(o, m): e for e, (o, m) in enumerate(zip(edge_op, edge_m))}
        makespan = label["makespan"]
        g["ref_makespan"] = makespan
        g["ref_status"] = label["status"]
        g["ref_start"] = starts
        g["target_edge"] = torch.tensor([edge_index[(o, machines[o])] for o in range(n)], dtype=torch.long)
        g["target_bin"] = torch.tensor(
            [min(PRIORITY_BINS - 1, int(starts[o] / makespan * PRIORITY_BINS)) for o in range(n)],
            dtype=torch.long)
    return g


def load_folder(folder, need_label=False):
    graphs, missing = [], 0
    for path in fjs_files(os.path.join(DATA_ROOT, folder)):
        label = read_label(path)
        if label is None:
            missing += 1
            if need_label:
                continue
        graphs.append(build_graph(read_fjs(path), label))
    return graphs, missing


train_graphs, val_graphs = [], []
for train_set in TRAIN_SETS:
    g_train, miss_train = load_folder(os.path.join(train_set, "train"), need_label=True)
    g_val, miss_val = load_folder(os.path.join(train_set, "val"), need_label=True)
    train_graphs += g_train
    val_graphs += g_val
    print(f"{train_set}: train {len(g_train)} labelled ({miss_train} without CP-SAT result), "
          f"val {len(g_val)} labelled ({miss_val} without CP-SAT result)")

if not train_graphs:
    print("No labelled training instances. Solve train/val with CP-SAT first (cell above).")

## Batching

Several graphs are glued into one big graph by shifting the node and edge numbers.

In [ ]:
class Batch:
    pass


def collate(graphs, with_target=True):
    b = Batch()
    op_x, m_x, edge_x = [], [], []
    edge_op, edge_m, edge_mlocal, n_elig, job_src, job_dst = [], [], [], [], [], []
    op_graph, m_graph, e_graph = [], [], []
    target_edge, target_bin = [], []
    op_off = m_off = e_off = 0
    for gi, g in enumerate(graphs):
        op_x.append(g["op_x"])
        m_x.append(g["m_x"])
        edge_x.append(g["edge_x"])
        edge_op.append(g["edge_op"] + op_off)
        edge_m.append(g["edge_m"] + m_off)
        edge_mlocal.append(g["edge_m"])
        n_elig.append(g["n_elig"])
        job_src.append(g["job_src"] + op_off)
        job_dst.append(g["job_dst"] + op_off)
        op_graph.append(torch.full((g["n"],), gi, dtype=torch.long))
        m_graph.append(torch.full((g["M"],), gi, dtype=torch.long))
        e_graph.append(torch.full((g["E"],), gi, dtype=torch.long))
        if with_target:
            target_edge.append(g["target_edge"] + e_off)
            target_bin.append(g["target_bin"])
        op_off += g["n"]
        m_off += g["M"]
        e_off += g["E"]

    b.G, b.n, b.M, b.E = len(graphs), op_off, m_off, e_off
    b.op_x = torch.cat(op_x).to(DEVICE)
    b.m_x = torch.cat(m_x).to(DEVICE)
    b.edge_x = torch.cat(edge_x).to(DEVICE)
    b.edge_op = torch.cat(edge_op).to(DEVICE)
    b.edge_m = torch.cat(edge_m).to(DEVICE)
    b.edge_mlocal = torch.cat(edge_mlocal).to(DEVICE)
    b.n_elig = torch.cat(n_elig).to(DEVICE)
    b.n_elig_edge = b.n_elig[b.edge_op]
    b.job_src = torch.cat(job_src).to(DEVICE)
    b.job_dst = torch.cat(job_dst).to(DEVICE)
    b.op_graph = torch.cat(op_graph).to(DEVICE)
    b.m_graph = torch.cat(m_graph).to(DEVICE)
    b.e_graph = torch.cat(e_graph).to(DEVICE)
    b.deg_m = torch.bincount(b.edge_m, minlength=b.M).clamp_min(1).float()
    b.target_edge = torch.cat(target_edge).to(DEVICE) if with_target else None
    b.target_bin = torch.cat(target_bin).to(DEVICE) if with_target else None
    return b

## Segment helpers

Each operation has a different number of eligible machines, so the machine choice is a softmax
over a variable-length group of edges. These helpers do max, log-sum-exp, and sampling per group.

In [ ]:
def segment_max(x, seg, n):
    out = torch.full((n,), float("-inf"), device=x.device, dtype=x.dtype)
    return out.scatter_reduce(0, seg, x, reduce="amax", include_self=True)


def segment_logsumexp(x, seg, n):
    mx = segment_max(x, seg, n)
    total = torch.zeros(n, device=x.device, dtype=x.dtype).index_add_(0, seg, torch.exp(x - mx[seg]))
    return mx + torch.log(total)


def segment_argmax(x, seg, n):
    best = segment_max(x, seg, n)
    hit = torch.nonzero(x == best[seg]).squeeze(1)
    out = torch.empty(n, dtype=torch.long, device=x.device)
    out[seg[hit]] = hit
    return out


def segment_sample(logp, seg, n):
    """Draw one edge per group with probability proportional to exp(logp)."""
    u = torch.rand_like(logp).clamp(1e-20, 1.0)
    return segment_argmax(logp - torch.log(-torch.log(u)), seg, n)

## Diffusion

Uniform categorical diffusion. With probability `alpha_bar[t]` a variable keeps its clean value;
otherwise it is replaced by a uniform random choice: a random eligible machine, or a random
priority bin. `alpha_bar` follows the cosine schedule and reaches 0 at `t = T`.

In [ ]:
steps_ = torch.arange(T + 1, dtype=torch.float64) / T
f_ = torch.cos((steps_ + 0.008) / 1.008 * math.pi / 2) ** 2
alpha_bar = (f_ / f_[0]).clamp(0.0, 1.0).float().to(DEVICE)
bin_centers = ((torch.arange(PRIORITY_BINS, dtype=torch.float) + 0.5) / PRIORITY_BINS).to(DEVICE)


def corrupt(b, t):
    """t has one noise level per graph. Returns noisy machine edges and priority bins."""
    keep_prob = alpha_bar[t][b.op_graph]
    random_edge = segment_sample(torch.zeros(b.E, device=DEVICE), b.edge_op, b.n)
    keep_m = torch.rand(b.n, device=DEVICE) < keep_prob
    machine_t = torch.where(keep_m, b.target_edge, random_edge)
    keep_r = torch.rand(b.n, device=DEVICE) < keep_prob
    random_bin = torch.randint(PRIORITY_BINS, (b.n,), device=DEVICE)
    priority_t = torch.where(keep_r, b.target_bin, random_bin)
    return machine_t, priority_t


def step_machine(b, machine_t, p_edge, t, s):
    """Sample machines at level s < t from the posterior given the model's clean prediction."""
    keep_ts = alpha_bar[t] / alpha_bar[s]
    a_s = alpha_bar[s]
    current = torch.zeros(b.E, device=DEVICE)
    current[machine_t] = 1.0
    k = b.n_elig_edge
    like = keep_ts * current + (1 - keep_ts) / k
    prior = a_s * p_edge + (1 - a_s) / k
    return segment_sample(torch.log(like * prior + 1e-20), b.edge_op, b.n)


def step_priority(priority_t, p_bin, t, s):
    keep_ts = alpha_bar[t] / alpha_bar[s]
    a_s = alpha_bar[s]
    like = keep_ts * F.one_hot(priority_t, PRIORITY_BINS).float() + (1 - keep_ts) / PRIORITY_BINS
    prior = a_s * p_bin + (1 - a_s) / PRIORITY_BINS
    post = like * prior
    return torch.multinomial(post / post.sum(-1, keepdim=True), 1).squeeze(1)

## Denoising network

Each layer passes messages along four relations: previous operation in the job, next operation in
the job, machine to operation, and operation to machine. Messages on operation-machine edges are
gated by the edge state, which includes whether the noisy sample currently puts the operation on
that machine. The diffusion step enters every layer through a scale and shift (FiLM).

Outputs: one logit per operation-machine edge (softmax over each operation's eligible machines),
and `PRIORITY_BINS` logits per operation.

In [ ]:
def time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    angles = t.float()[:, None] * freqs[None]
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class Film(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.norm = nn.LayerNorm(h)
        self.cond = nn.Linear(h, 2 * h)

    def forward(self, x, cond, index):
        scale, shift = self.cond(cond)[index].chunk(2, dim=-1)
        return self.norm(x) * (1 + scale) + shift


class Layer(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.op_self = nn.Linear(h, h)
        self.prev_msg = nn.Linear(h, h)
        self.next_msg = nn.Linear(h, h)
        self.m_to_op = nn.Linear(h, h)
        self.m_self = nn.Linear(h, h)
        self.op_to_m = nn.Linear(h, h)
        self.e_self = nn.Linear(h, h)
        self.e_op = nn.Linear(h, h)
        self.e_m = nn.Linear(h, h)
        self.film_op = Film(h)
        self.film_m = Film(h)
        self.film_e = Film(h)

    def forward(self, ho, hm, he, cond, b):
        gate = torch.sigmoid(he)

        msg_op = torch.zeros_like(ho)
        msg_op.index_add_(0, b.job_dst, self.prev_msg(ho)[b.job_src])
        msg_op.index_add_(0, b.job_src, self.next_msg(ho)[b.job_dst])
        from_m = torch.zeros_like(ho).index_add_(0, b.edge_op, gate * self.m_to_op(hm)[b.edge_m])
        msg_op = msg_op + from_m / b.n_elig[:, None]

        from_op = torch.zeros_like(hm).index_add_(0, b.edge_m, gate * self.op_to_m(ho)[b.edge_op])
        msg_m = from_op / b.deg_m[:, None]

        new_ho = ho + F.relu(self.film_op(self.op_self(ho) + msg_op, cond, b.op_graph))
        new_hm = hm + F.relu(self.film_m(self.m_self(hm) + msg_m, cond, b.m_graph))
        new_he = he + F.relu(self.film_e(
            self.e_self(he) + self.e_op(new_ho)[b.edge_op] + self.e_m(new_hm)[b.edge_m], cond, b.e_graph))
        return new_ho, new_hm, new_he


class Denoiser(nn.Module):
    def __init__(self, h=HIDDEN, layers=LAYERS, bins=PRIORITY_BINS):
        super().__init__()
        self.h = h
        self.op_in = nn.Linear(6, h)
        self.priority_in = nn.Embedding(bins, h)
        self.m_in = nn.Linear(2, h)
        self.edge_in = nn.Linear(2, h)
        self.assigned_in = nn.Embedding(2, h)
        self.time_mlp = nn.Sequential(nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h))
        self.layers = nn.ModuleList([Layer(h) for _ in range(layers)])
        self.machine_out = nn.Sequential(nn.Linear(3 * h, h), nn.SiLU(), nn.Linear(h, 1))
        self.priority_out = nn.Sequential(nn.Linear(h, h), nn.SiLU(), nn.Linear(h, bins))

    def forward(self, b, machine_t, priority_t, t):
        assigned = torch.zeros(b.E, dtype=torch.long, device=DEVICE)
        assigned[machine_t] = 1
        ho = self.op_in(b.op_x) + self.priority_in(priority_t)
        hm = self.m_in(b.m_x)
        he = self.edge_in(b.edge_x) + self.assigned_in(assigned)
        cond = self.time_mlp(time_embedding(t, self.h))
        for layer in self.layers:
            ho, hm, he = layer(ho, hm, he, cond, b)
        machine_logits = self.machine_out(torch.cat([he, ho[b.edge_op], hm[b.edge_m]], dim=-1)).squeeze(-1)
        priority_logits = self.priority_out(ho)
        return machine_logits, priority_logits


model = Denoiser().to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## From machines and priorities to a schedule

`decode` is list scheduling. `decode_nondelay` ("no idle machine") uses the same machines and
priorities, but at each step only the operations that can start the earliest are candidates, and
the lowest priority among them goes first, so a machine never waits while an operation is ready for
it. `DECODERS` maps the names used in `CONFIG` to these functions. `check` verifies a schedule independently: every operation runs on an
eligible machine, operations of a job run in order, and no machine runs two operations at once.

In [ ]:
def decode(g, machines, priority):
    n_jobs = len(g["job_first"])
    next_op = list(g["job_first"])
    job_end = [g["job_first"][j] + g["job_len"][j] for j in range(n_jobs)]
    job_free = [0] * n_jobs
    machine_free = [0] * g["M"]
    start = [0] * g["n"]
    active = list(range(n_jobs))
    for _ in range(g["n"]):
        j = min(active, key=lambda j: (priority[next_op[j]], next_op[j]))
        o = next_op[j]
        m = machines[o]
        s = max(job_free[j], machine_free[m])
        start[o] = s
        job_free[j] = machine_free[m] = s + g["options"][o][m]
        next_op[j] += 1
        if next_op[j] == job_end[j]:
            active.remove(j)
    return max(job_free), start


def decode_nondelay(g, machines, priority):
    n_jobs = len(g["job_first"])
    next_op = list(g["job_first"])
    job_end = [g["job_first"][j] + g["job_len"][j] for j in range(n_jobs)]
    job_free = [0] * n_jobs
    machine_free = [0] * g["M"]
    start = [0] * g["n"]
    active = list(range(n_jobs))
    for _ in range(g["n"]):
        # (job, operation, earliest start) for the next operation of every unfinished job
        heads = [(j, next_op[j], max(job_free[j], machine_free[machines[next_op[j]]])) for j in active]
        earliest = min(s for _, _, s in heads)
        j, o, s = min((h for h in heads if h[2] == earliest), key=lambda h: (priority[h[1]], h[1]))
        m = machines[o]
        start[o] = s
        job_free[j] = machine_free[m] = s + g["options"][o][m]
        next_op[j] += 1
        if next_op[j] == job_end[j]:
            active.remove(j)
    return max(job_free), start


DECODERS = {"current": decode, "nondelay": decode_nondelay}


def check(g, machines, start):
    for o in range(g["n"]):
        if machines[o] not in g["options"][o]:
            return False
    for j, first in enumerate(g["job_first"]):
        for o in range(first, first + g["job_len"][j] - 1):
            if start[o] + g["options"][o][machines[o]] > start[o + 1]:
                return False
    by_machine = {}
    for o in range(g["n"]):
        by_machine.setdefault(machines[o], []).append((start[o], start[o] + g["options"][o][machines[o]]))
    for intervals in by_machine.values():
        intervals.sort()
        for (_, end_a), (start_b, _) in zip(intervals, intervals[1:]):
            if end_a > start_b:
                return False
    return True

## Sanity check: how much do the priority bins lose?

Decode each training label with CP-SAT's machines and (1) the exact start times, then (2) the
binned priorities the model is trained on. (1) should give a 0% gap. (2) shows how much is lost by
binning. If it is large, increase `PRIORITY_BINS`.

In [ ]:
def label_gap(graphs, limit=200):
    exact, binned, feasible = [], [], True
    for g in graphs[:limit]:
        machines = g["edge_m"][g["target_edge"]].tolist()
        span_exact, start = decode(g, machines, g["ref_start"])
        feasible &= check(g, machines, start)
        span_bin, start = decode(g, machines, bin_centers[g["target_bin"].to(DEVICE)].tolist())
        feasible &= check(g, machines, start)
        exact.append(span_exact / g["ref_makespan"] - 1)
        binned.append(span_bin / g["ref_makespan"] - 1)
    return 100 * np.mean(exact), 100 * np.mean(binned), feasible


if train_graphs:
    gap_exact, gap_binned, ok = label_gap(train_graphs)
    print(f"exact start times: {gap_exact:.2f}% gap | {PRIORITY_BINS} bins: {gap_binned:.2f}% gap | all feasible: {ok}")

## Generation

Start from random eligible machines and random priority bins, then denoise in `SAMPLE_STEPS`
steps. All samples of one instance are generated in one batch. At the end, each operation takes the
most likely machine and the expected priority from the last prediction. The same samples are then
turned into schedules by each decoder in `decoders`; results are reported per decoder.

In [ ]:
@torch.no_grad()
def generate(model, g, samples, decoders=("current",)):
    """Returns {decoder: [(makespan, feasible) per sample]} and {"sample" / decoder: seconds}."""
    t0 = time.time()
    model.eval()
    b = collate([g] * samples, with_target=False)
    machine = segment_sample(torch.zeros(b.E, device=DEVICE), b.edge_op, b.n)
    priority = torch.randint(PRIORITY_BINS, (b.n,), device=DEVICE)
    levels = torch.linspace(T, 0, SAMPLE_STEPS + 1).round().long().tolist()
    for t, s in zip(levels[:-1], levels[1:]):
        t_batch = torch.full((samples,), t, device=DEVICE, dtype=torch.long)
        machine_logits, priority_logits = model(b, machine, priority, t_batch)
        if s == 0:
            break
        p_edge = torch.exp(machine_logits - segment_logsumexp(machine_logits, b.edge_op, b.n)[b.edge_op])
        p_bin = torch.softmax(priority_logits, dim=-1)
        machine = step_machine(b, machine, p_edge, t, s)
        priority = step_priority(priority, p_bin, t, s)

    chosen = segment_argmax(machine_logits, b.edge_op, b.n)
    machines = b.edge_mlocal[chosen].view(samples, g["n"]).tolist()
    priorities = (torch.softmax(priority_logits, dim=-1) @ bin_centers).view(samples, g["n"]).tolist()
    seconds = {"sample": time.time() - t0}
    results = {}
    for name in decoders:
        t0 = time.time()
        results[name] = []
        for k in range(samples):
            span, start = DECODERS[name](g, machines[k], priorities[k])
            results[name].append((span, check(g, machines[k], start)))
        seconds[name] = time.time() - t0
    return results, seconds


def evaluate(model, graphs, samples, limit=None, decoders=("current",)):
    rows = []
    for g in graphs[:limit]:
        results, seconds = generate(model, g, samples, decoders)
        row = {"instance": g["name"], "cpsat": g["ref_makespan"], "cpsat_status": g["ref_status"]}
        for name in decoders:
            spans = [span for span, _ in results[name]]
            row.update({
                f"{name}_first": spans[0],
                f"{name}_mean": float(np.mean(spans)),
                f"{name}_best": min(spans),
                f"{name}_feasible": all(ok for _, ok in results[name]),
                f"{name}_seconds": seconds["sample"] + seconds[name],
            })
        rows.append(row)
    return rows


def summarize(rows, decoder="current"):
    with_ref = [r for r in rows if r["cpsat"]]
    gap = lambda key: 100 * float(np.mean([r[key] / r["cpsat"] - 1 for r in with_ref])) if with_ref else float("nan")
    return {
        "instances": len(rows),
        "gap_first_%": gap(f"{decoder}_first"),
        "gap_mean_%": gap(f"{decoder}_mean"),
        "gap_best_%": gap(f"{decoder}_best"),
        "feasible_%": 100 * float(np.mean([r[f"{decoder}_feasible"] for r in rows])),
        "seconds_per_instance": float(np.mean([r[f"{decoder}_seconds"] for r in rows])),
    }

## Training

One step: pick a noise level per instance, corrupt the CP-SAT machines and priorities, and train
the network to predict the clean ones (cross-entropy on both). Every `EVAL_EVERY` epochs we generate
schedules for part of the validation set; the checkpoint with the lowest best-of-`VAL_SAMPLES` gap
is saved to `OUT_DIR/best.pt`.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = max(1, math.ceil(len(train_graphs) / BATCH_SIZE))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * steps_per_epoch)
history = []
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_graphs)
    sums = {"loss": 0.0, "machine": 0.0, "priority": 0.0}
    t0 = time.time()
    for i in range(0, len(train_graphs), BATCH_SIZE):
        b = collate(train_graphs[i:i + BATCH_SIZE])
        t = torch.randint(1, T + 1, (b.G,), device=DEVICE)
        machine_t, priority_t = corrupt(b, t)
        machine_logits, priority_logits = model(b, machine_t, priority_t, t)

        lse = segment_logsumexp(machine_logits, b.edge_op, b.n)
        loss_machine = (lse - machine_logits[b.target_edge]).mean()
        loss_priority = F.cross_entropy(priority_logits, b.target_bin)
        loss = loss_machine + PRIORITY_WEIGHT * loss_priority

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        sums["loss"] += loss.item()
        sums["machine"] += loss_machine.item()
        sums["priority"] += loss_priority.item()

    record = {"epoch": epoch, **{k: v / steps_per_epoch for k, v in sums.items()}}
    line = (f"epoch {epoch:3d} | loss {record['loss']:.4f} (machine {record['machine']:.4f}, "
            f"priority {record['priority']:.4f}) | {time.time() - t0:.1f}s")

    if val_graphs and (epoch % EVAL_EVERY == 0 or epoch == EPOCHS):
        val = summarize(evaluate(model, val_graphs, VAL_SAMPLES, VAL_LIMIT))
        record.update({f"val_{k}": v for k, v in val.items()})
        line += f" | val gap first {val['gap_first_%']:.2f}% best {val['gap_best_%']:.2f}%"
        if val["gap_best_%"] < best_val:
            best_val = val["gap_best_%"]
            torch.save({"model": model.state_dict(), "epoch": epoch, "settings": CONFIG}, os.path.join(OUT_DIR, "best.pt"))
            line += " (saved)"
    history.append(record)
    print(line)

with open(os.path.join(OUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=1)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
epochs = [h["epoch"] for h in history]
axes[0].plot(epochs, [h["machine"] for h in history], label="machine")
axes[0].plot(epochs, [h["priority"] for h in history], label="priority")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training loss")
axes[0].legend()
val_points = [h for h in history if "val_gap_best_%" in h]
axes[1].plot([h["epoch"] for h in val_points], [h["val_gap_first_%"] for h in val_points], "o-", label="first sample")
axes[1].plot([h["epoch"] for h in val_points], [h["val_gap_best_%"] for h in val_points], "o-",
             label=f"best of {VAL_SAMPLES}")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("validation gap to CP-SAT (%)")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training.png"), dpi=150)
plt.show()

## Evaluation

Loads the best checkpoint and evaluates every folder in `EVAL_SETS`. For each instance we report
the first sample, the mean over samples, and the best of `SAMPLES`, each as a gap to the CP-SAT
makespan, once for every decoder in `DECODERS` (same samples). Per-instance results go to
`OUT_DIR/eval/`; `eval_summary.json` has one entry per folder and decoder.

In [ ]:
checkpoint = torch.load(os.path.join(OUT_DIR, "best.pt"), map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
print("loaded checkpoint from epoch", checkpoint["epoch"])

eval_dir = os.path.join(OUT_DIR, "eval")
os.makedirs(eval_dir, exist_ok=True)
summary = {}
for folder in EVAL_SETS:
    graphs, missing = load_folder(folder)
    if not graphs:
        print(f"{folder}: no instances found")
        continue
    rows = evaluate(model, graphs, SAMPLES, decoders=CONFIG["DECODERS"])
    summary[folder] = {name: summarize(rows, name) for name in CONFIG["DECODERS"]}
    out = os.path.join(eval_dir, folder.replace("/", "__") + ".csv")
    with open(out, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    note = f" ({missing} without CP-SAT result)" if missing else ""
    for name, s in summary[folder].items():
        print(f"{folder}{note} [{name}]: first {s['gap_first_%']:.2f}% | mean {s['gap_mean_%']:.2f}% | "
              f"best of {SAMPLES} {s['gap_best_%']:.2f}% | feasible {s['feasible_%']:.0f}% | "
              f"{s['seconds_per_instance']:.2f}s per instance")

with open(os.path.join(OUT_DIR, "eval_summary.json"), "w") as f:
    json.dump(summary, f, indent=1)